# Tactile Segmentation Colab Pipeline

这个 Notebook 适用于 **Google Colab**，包含：
1. 环境安装
2. 代码/数据准备
3. 训练（支持断点续训）
4. 导出（CoreML / TFLite，按依赖可用性自动跳过）
5. 下载模型产物


## 0) 可选：挂载 Google Drive
如果你希望保存 checkpoint 和数据到 Drive，请取消注释后执行。


In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# %cd /content/drive/MyDrive


## 1) 准备项目代码
方式 A：直接 `git clone` 你的仓库（推荐）。

方式 B：如果你只上传了本 `.ipynb`，请先把代码文件放到 `/content/visual`。


In [ ]:
# === 方式 A: clone 仓库（把 REPO_URL 换成你的仓库）===
REPO_URL = ''  # e.g. 'https://github.com/yourname/visual.git'
BRANCH = ''    # 可选: 'main' 或你的分支名

import os, subprocess

if REPO_URL:
    if os.path.exists('/content/visual'):
        subprocess.run(['rm', '-rf', '/content/visual'], check=False)
    cmd = ['git', 'clone', REPO_URL, '/content/visual']
    if BRANCH:
        cmd = ['git', 'clone', '-b', BRANCH, REPO_URL, '/content/visual']
    subprocess.run(cmd, check=True)

%cd /content/visual
!pwd


## 2) 安装依赖
包含训练、数据处理和可选导出依赖。


In [ ]:
!python -m pip install -U pip
!pip install pyyaml tqdm albumentations scikit-learn opencv-python Pillow numpy imagehash icrawler gradio>=4.0
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121
# 可选导出依赖（失败可忽略，训练不受影响）
!pip install coremltools onnx onnx-tf tensorflow || true


## 3) (可选) 抓取图片
如果你已经有 `dataset/images` 和 `dataset/masks`，可跳过。


In [ ]:
MAX_IMAGES = 500
!python scraper.py --max_images {MAX_IMAGES}


## 4) 准备标注数据
训练前需要保证 `dataset/masks` 中存在与图像同名的 `.png` mask（像素 0/1/2）。

你可以在本地运行 `annotator.py` 完成标注后再把数据上传回 Colab。


In [ ]:
from pathlib import Path
img_dir = Path('dataset/images')
mask_dir = Path('dataset/masks')
imgs = [p for p in img_dir.glob('*') if p.suffix.lower() in {'.jpg','.jpeg','.png'}]
pairs = [p for p in imgs if (mask_dir / f'{p.stem}.png').exists()]
print(f'images={len(imgs)}, paired_masks={len(pairs)}')
if len(pairs) == 0:
    raise RuntimeError('没有可训练的 image-mask 对，请先完成标注。')


## 5) 配置训练参数
按需修改 `train/config.yaml`。


In [ ]:
import yaml
from pathlib import Path
cfg_path = Path('train/config.yaml')
cfg = yaml.safe_load(cfg_path.read_text(encoding='utf-8'))
cfg['training']['epochs'] = 30
cfg['training']['batch_size'] = 8
cfg['training']['num_workers'] = 2
cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True), encoding='utf-8')
print(cfg_path.read_text(encoding='utf-8'))


## 6) 开始训练


In [ ]:
!python train/train.py --config train/config.yaml


## 7) 断点续训（可选）


In [ ]:
!python train/train.py --config train/config.yaml --resume checkpoints/last_checkpoint.pth


## 8) 查看并下载产物
训练 best checkpoint 与导出文件在 `checkpoints/` 和 `exports/`。


In [ ]:
!ls -lah checkpoints || true
!ls -lah exports || true

from google.colab import files
# files.download('checkpoints/best_lraspp.pth')
# files.download('exports/tactile_seg_int8.tflite')
# files.download('exports/tactile_seg.mlmodel')
